In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 94 — Tool-Selection Benchmark

## Goal
Evaluate how often an LLM agent **picks the correct tool**
when given ambiguous queries. Build an eval framework to
measure tool-selection accuracy.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Tool binding** | `bind_tools` attaches tool schemas to LLM |
| **Eval design** | Ground-truth labels + automated scoring |
| **Ambiguity handling** | Testing edge cases where tools overlap |
| **Metrics** | Accuracy, confusion matrix, per-tool precision |

## Stack
- `langchain-ollama` — LLM with tool calling
- `langchain-core` — tool definitions
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
from collections import Counter
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Define a Tool Suite

We create 6 tools that have **overlapping** domains to test
the agent's ability to pick the right one.

In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the public internet for general information, news, and facts."""
    return f"Web results for: {query}"

@tool
def search_internal_docs(query: str) -> str:
    """Search internal company documents, policies, and procedures."""
    return f"Internal docs for: {query}"

@tool
def query_database(sql: str) -> str:
    """Run a SQL query against the company's sales and customer database."""
    return f"DB results for: {sql}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a person. Requires recipient, subject, and body."""
    return f"Email sent to {to}"

@tool
def create_calendar_event(title: str, date: str, attendees: str) -> str:
    """Create a calendar meeting or event with a title, date, and attendees."""
    return f"Event '{title}' created"

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression or perform calculations."""
    return f"Result: {expression}"

all_tools = [search_web, search_internal_docs, query_database,
             send_email, create_calendar_event, calculate]

llm_with_tools = llm.bind_tools(all_tools)

print(f"Defined {len(all_tools)} tools: {[t.name for t in all_tools]}")

## Step 2 — Evaluation Dataset

Each test case has a query and the expected (ground truth) tool.
Some queries are deliberately ambiguous.

In [ ]:
EVAL_DATASET = [
    # Clear cases
    {"query": "What is the latest news about OpenAI?", "expected": "search_web",
     "difficulty": "easy"},
    {"query": "What is our company's remote work policy?", "expected": "search_internal_docs",
     "difficulty": "easy"},
    {"query": "How many customers signed up last month?", "expected": "query_database",
     "difficulty": "easy"},
    {"query": "Send a thank-you note to john@acme.com", "expected": "send_email",
     "difficulty": "easy"},
    {"query": "Schedule a team standup for Monday at 9am", "expected": "create_calendar_event",
     "difficulty": "easy"},
    {"query": "What is 15% of $24,500?", "expected": "calculate",
     "difficulty": "easy"},
    
    # Ambiguous cases (could be multiple tools)
    {"query": "Find the total revenue for Q3 2024", "expected": "query_database",
     "difficulty": "medium", "note": "Could be web or DB, but company data → DB"},
    {"query": "What's the industry standard for employee churn rate?", "expected": "search_web",
     "difficulty": "medium", "note": "Could be internal docs, but 'industry standard' → web"},
    {"query": "Show me the expense approval process", "expected": "search_internal_docs",
     "difficulty": "medium", "note": "Internal process → internal docs"},
    {"query": "Calculate the year-over-year growth from $1.2M to $1.5M", "expected": "calculate",
     "difficulty": "medium"},
    
    # Tricky cases
    {"query": "Check if we have a vendor agreement with Acme Corp", "expected": "search_internal_docs",
     "difficulty": "hard", "note": "Could be DB, but agreements → docs"},
    {"query": "Let Sarah know the proposal is ready and set up a review call", "expected": "send_email",
     "difficulty": "hard", "note": "Multi-action: email + calendar. Primary = email"},
    {"query": "How does our sales compare to Salesforce benchmarks?", "expected": "search_web",
     "difficulty": "hard", "note": "Needs external benchmark → web"},
    {"query": "Pull the customer list and count how many are enterprise tier", "expected": "query_database",
     "difficulty": "hard", "note": "'Pull' + 'count' → database"},
    {"query": "What's the margin if we discount 20% on a $50K deal?", "expected": "calculate",
     "difficulty": "hard"},
]

print(f"Evaluation dataset: {len(EVAL_DATASET)} test cases")
difficulties = Counter(tc['difficulty'] for tc in EVAL_DATASET)
for d, c in sorted(difficulties.items()):
    print(f"  {d}: {c} cases")

## Step 3 — Run the Benchmark

In [ ]:
def run_benchmark(dataset, llm_tools):
    """Run tool selection benchmark and collect results."""
    results = []
    
    for i, tc in enumerate(dataset):
        query = tc["query"]
        expected = tc["expected"]
        
        response = llm_tools.invoke([
            SystemMessage(content="You are a helpful assistant with access to tools. "
                          "Use the most appropriate tool for each request."),
            HumanMessage(content=query)
        ])
        
        # Extract selected tool
        if response.tool_calls:
            selected = response.tool_calls[0]["name"]
        else:
            selected = "NO_TOOL"
        
        correct = selected == expected
        icon = "✅" if correct else "❌"
        print(f"  {icon} [{tc['difficulty']:6s}] {query[:50]:50s} → {selected:25s} (expected: {expected})")
        
        results.append({
            "query": query,
            "expected": expected,
            "selected": selected,
            "correct": correct,
            "difficulty": tc["difficulty"]
        })
    
    return results

print("Running benchmark...\n")
results = run_benchmark(EVAL_DATASET, llm_with_tools)

## Step 4 — Analyze Results

In [ ]:
# Overall accuracy
correct = sum(1 for r in results if r["correct"])
total = len(results)
print(f"\nOverall Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")

# Accuracy by difficulty
print("\nAccuracy by Difficulty:")
for diff in ["easy", "medium", "hard"]:
    subset = [r for r in results if r["difficulty"] == diff]
    if subset:
        acc = sum(1 for r in subset if r["correct"]) / len(subset)
        print(f"  {diff:8s}: {100*acc:.1f}% ({sum(1 for r in subset if r['correct'])}/{len(subset)})")

# Per-tool precision
print("\nPer-Tool Results:")
tool_names = set(r["expected"] for r in results)
for tn in sorted(tool_names):
    expected_as = [r for r in results if r["expected"] == tn]
    selected_as = [r for r in results if r["selected"] == tn]
    correct_for = sum(1 for r in expected_as if r["correct"])
    print(f"  {tn:25s}  recall={correct_for}/{len(expected_as)}")

# Confusion analysis
print("\nMisclassifications:")
mistakes = [r for r in results if not r["correct"]]
if not mistakes:
    print("  None! Perfect score.")
for m in mistakes:
    print(f"  ❌ Expected '{m['expected']}' got '{m['selected']}': {m['query'][:60]}")

In [ ]:
# Build confusion matrix
all_tool_names = sorted(set([r["expected"] for r in results] + [r["selected"] for r in results]))
matrix = {e: Counter() for e in all_tool_names}
for r in results:
    matrix[r["expected"]][r["selected"]] += 1

print("Confusion Matrix (rows=expected, cols=predicted):\n")
header = f"{'':25s}" + "".join(f"{t[:8]:>10s}" for t in all_tool_names)
print(header)
print("-" * len(header))
for expected in all_tool_names:
    row = f"{expected:25s}"
    for predicted in all_tool_names:
        count = matrix[expected][predicted]
        cell = f"{count:>10d}" if count > 0 else f"{'·':>10s}"
        row += cell
    print(row)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Tool binding** | `llm.bind_tools(tools)` |
| **Ground truth** | Each query has a known correct tool |
| **Difficulty tiers** | Easy, medium, hard to test ambiguity |
| **Metrics** | Accuracy, per-tool recall, confusion matrix |

## Why This Matters
- Tool-selection errors are a major failure mode in agentic systems
- Ambiguous queries expose model weaknesses
- Systematic evaluation > vibes-based testing
- Confusion matrices show which tools get confused

## Extending This Project
- Test multiple models (compare Qwen vs Llama vs Mistral)
- Add more ambiguous test cases
- Test with different prompt templates
- Multi-tool calls (when query needs 2+ tools)
- Track latency alongside accuracy